# CarMaker-based Autonomous Driving Validation Framework

**Author:** Seungji Ryu (seungjiryu@hanyang.ac.kr)  
**Course:** Final Project  

This notebook walks through the setup and execution of the CarMaker-based autonomous driving validation framework.  
The system supports two ML-based M2M planning models inside a full ROS Noetic autonomous driving stack,
validated in closed-loop against a CarMaker high-fidelity vehicle simulator:

- **Diffusion Planner** — DiT-based trajectory generation with optional classifier guidance (collision avoidance, lane keeping)
- **Pluto** — transformer-based multi-modal trajectory planner (nuPlan-origin, adapted for CarMaker without nuPlan dependencies)

The active model is selected at launch time via `--model diffusion_planner` (default) or `--model pluto`.

> **Note:** Active system execution requires a running CarMaker instance and a machine with ROS Noetic + CUDA 11.8.  
> Cells that start the ROS stack are marked **[SHELL]** and should be run in a terminal on the target machine.

---
## 1. Repository Structure

```
final-project-SeungjiRyu/
├── config/                               # Runtime configuration (INI, YAML)
│   ├── planning_ml.yaml                  # Model selection, checkpoint paths, guidance settings
│   ├── rosparam_system.yaml              # Task periods for every ROS node
│   └── perception.ini / planning.ini / control.ini / ...
├── install/                              # Pre-built binary ROS packages (lab internal)
├── launch/
│   ├── simulator.launch                  # CarMaker bridge
│   └── rte.launch                        # TF transforms
├── src/
│   ├── mid2mid_planning_diffusion/       # ← Diffusion Planner ROS node (source)
│   │   ├── scripts/mid2mid_planning_diffusion_node.py
│   │   └── src/diffusion_planner/
│   │       ├── diffusion_planner_algorithm.py
│   │       └── diffusion_planner/
│   │           ├── model/               # DiT backbone + guidance functions
│   │           └── data_process/        # Coordinate transforms, normalizer
│   └── mid2mid_planning_pluto/           # ← Pluto ROS node (source)
│       ├── scripts/mid2mid_planning_pluto_node.py
│       └── src/pluto/
│           ├── pluto_algorithm.py        # PlutoAlgorithm: PlanningSpace → Trajectory
│           ├── pluto_model.py            # PlutoPlanningModel (nn.Module, no nuPlan)
│           ├── layers/                   # Fourier embedding, transformer, MLP, NAT
│           └── modules/                  # Agent/map encoder, predictor, decoder
├── nuplan-devkit/                        # Upstream nuPlan SDK (offline evaluation only)
├── launch_carmaker.sh                    # One-command system launcher (--model flag)
└── install_hpipm.sh                      # HPIPM / blasfeo build script
```

**What is in source vs. binary:**  
Only the M2M planning nodes (`src/mid2mid_planning_diffusion/`, `src/mid2mid_planning_pluto/`) and configuration files are provided as source.  
All other lab-internal ROS packages ship as pre-built binaries in `install/`.

---
## 2. Environment Setup

### 2.1 System Requirements

| Item | Requirement |
|------|-------------|
| OS | Ubuntu 20.04 LTS |
| ROS | Noetic |
| Python | 3.9 |
| CUDA | 11.8 |
| CarMaker | ROS Noetic compatible version |

### 2.2 Python Virtual Environment

The Diffusion Planner node uses a dedicated Python 3.9 venv at `/venv/diffusion_planner`
(not the system Python or conda).  The node script's shebang points to this venv directly.

In [ ]:
# [SHELL] Create and populate the Python venvs

setup_diffusion = [
    "sudo apt install -y python3.9-venv",
    "sudo mkdir -p /venv && sudo python3.9 -m venv /venv/diffusion_planner && sudo chown -R $USER:$USER /venv/diffusion_planner",
    "source /venv/diffusion_planner/bin/activate && pip install -e src/mid2mid_planning_diffusion/src/diffusion_planner && pip install -r src/mid2mid_planning_diffusion/src/diffusion_planner/requirements.txt",
]

setup_pluto = [
    "sudo python3.9 -m venv /venv/pluto && sudo chown -R $USER:$USER /venv/pluto",
    "source /venv/pluto/bin/activate && pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118",
    "pip install natten timm rospkg netifaces numpy",
]

print('--- Diffusion Planner venv ---')
for cmd in setup_diffusion:
    print(f'$ {cmd}')

print('\n--- Pluto venv ---')
for cmd in setup_pluto:
    print(f'$ {cmd}')

In [ ]:
# [SHELL] Build HPIPM / blasfeo (QP solver backend for the MPC controller)
print("$ ./install_hpipm.sh")

In [ ]:
# [SHELL] Source the pre-built ROS workspace
print("$ source install/setup.bash")

---
## 3. Configuration

### 3.1 ML Planner settings (`config/planning_ml.yaml`)

This file controls **model selection**, checkpoint paths, and guidance settings for both models.

In [ ]:
import yaml, os

config_path = os.path.join(os.getcwd(), "config", "planning_ml.yaml")
with open(config_path) as f:
    ml_config = yaml.safe_load(f)

import json
print(json.dumps(ml_config, indent=2))

### Model Selection

Switch the active planning model at **launch time** — no config file edit needed:

```bash
./launch_carmaker.sh --model diffusion_planner   # default
./launch_carmaker.sh --model pluto
```

Both models subscribe to `app/pla/planning_space` and publish to `app/pla/trajectory` at 10 Hz.

### Classifier Guidance (Diffusion Planner only)

Guidance can be toggled without any code change:

```python
ml_config["diffusion_planner"]["use_collision_guidance"] = True   # collision avoidance
ml_config["diffusion_planner"]["lane_keeping"]["enable"] = True   # lane keeping
```

Then write the updated config back to `config/planning_ml.yaml` before launching.

In [ ]:
import configparser

ini_path = os.path.join(os.getcwd(), "config", "rosparam_system.yaml")
with open(ini_path) as f:
    rosparam = yaml.safe_load(f)

periods = rosparam["task_period"]
print(f"Planning node period : {periods['period_mid2mid_planning_ml_node']} s  ({1/periods['period_mid2mid_planning_ml_node']:.0f} Hz)")
print(f"Control period       : {periods['period_lateral_control']} s  ({1/periods['period_lateral_control']:.0f} Hz)")
print(f"Perception period    : {periods['period_motion_prediction']} s  ({1/periods['period_motion_prediction']:.0f} Hz)")

---
## 4. Planning Model Pipelines

Both models share the same three-stage inference interface: **RunPreprocess → RunAlgorithm → RunPostprocess**.

### 4.1 Diffusion Planner

Implemented in `src/mid2mid_planning_diffusion/src/diffusion_planner/diffusion_planner_algorithm.py`.

### 4.2 Pluto

Implemented in `src/mid2mid_planning_pluto/src/pluto/pluto_algorithm.py`.  
The `PlutoAlgorithm` class converts `PlanningSpace` messages directly to Pluto's tensor format without any nuPlan runtime.

In [ ]:
# Three-stage inference pipeline — shared interface for both models

pipelines = {
    "Diffusion Planner": [
        ("1. RunPreprocess",
         "PlanningSpace (ROS msg)",
         ["Parse ego, agents, statics, lanes, routes",
          "Transform to ego-centric frame",
          "Truncate: 32 agents, 70 lanes, 25 routes",
          "Z-score normalise"],
         "Normalised tensor dict + PreprocessData"),
        ("2. RunAlgorithm",
         "Normalised tensor dict",
         ["Forward pass: DiT backbone (VP-SDE + DPM-Solver)",
          "Optional: classifier guidance gradients per denoising step"],
         "ego_prediction (T+1,4)  +  agent_predictions (N,T+1,4) — [x,y,cosθ,sinθ]"),
        ("3. RunPostprocess",
         "DiffusionPlannerOutput + PreprocessData",
         ["Inverse transform to global frame",
          "Delay compensation via linear interpolation",
          "Truncate to future_steps (default 80 = 8 s)",
          "Pack into autohyu_msgs/Trajectory"],
         "Trajectory msg  +  PredictObjects msg"),
    ],
    "Pluto": [
        ("1. RunPreprocess",
         "PlanningSpace (ROS msg)",
         ["Parse ego history (21 steps), agents, map polygons, static objects, routes",
          "Concatenate route segments into a single reference polyline (100 pts)",
          "Transform all features to ego-centric frame",
          "Filter map polygons within radius=100 m"],
         "Tensor dict (bs=1) + preprocess_meta"),
        ("2. RunAlgorithm",
         "Tensor dict",
         ["Agent encoding via Neighborhood Attention (NAT)",
          "Map polygon encoding via cross-attention",
          "Multi-modal trajectory decoding (6 modes)"],
         "output_trajectory (T,3)  +  candidate_trajectories (6,T,3)"),
        ("3. RunPostprocess",
         "PlutoOutput + preprocess_meta",
         ["Select best-mode trajectory (output_trajectory)",
          "Inverse transform to global frame",
          "Compute velocity from position differences",
          "Pack into autohyu_msgs/Trajectory"],
         "Trajectory msg"),
    ],
}

for model_name, stages in pipelines.items():
    print(f'\n{'='*65}')
    print(f'  {model_name}')
    print(f'{'='*65}')
    for stage, inp, steps, out in stages:
        print(f'  {stage}')
        print(f'  Input : {inp}')
        for s in steps:
            print(f'    • {s}')
        print(f'  Output: {out}')
        print()

### Input tensor shapes (after fixed-size conversion)

| Key | Shape | Description |
|-----|-------|-------------|
| `neighbor_agents_past` | `(1, 32, T_hist, 11)` | Agent history: x, y, cosθ, sinθ, vx, vy, w, l, type×3 |
| `ego_current_state` | `(1, 10)` | Ego: x, y, cosθ, sinθ, vx, vy, ax, ay, steer, yaw_rate |
| `static_objects` | `(1, N_s, 10)` | Static obstacles |
| `lanes` | `(1, 70, 20, 12)` | Lane polylines: centerline, vector, left/right offset, traffic light×4 |
| `route_lanes` | `(1, 25, 20, 12)` | Route polylines (same format as lanes) |

---
## 5. Launching the Full System

The entire stack is launched with a single shell script.  
Use `--model` to select the ML planner (default: `diffusion_planner`).  
The script generates a Terminator layout and starts all 13 modules in the correct order.

In [ ]:
launch_sequence = [
    ("RViz",                   "rviz -d resources/rviz/validation_framework.rviz"),
    ("CarMaker bridge",        "roslaunch launch/simulator.launch"),
    ("TF (RTE)",               "roslaunch launch/rte.launch"),
    ("Motion prediction",      "roslaunch motion_prediction motion_prediction.launch"),
    ("Map loader",             "roslaunch map_loader map_loader.launch location:=<map>"),
    ("Goalpoint planning",     "roslaunch goalpoint_planning goalpoint_planning.launch"),
    ("Global planning",        "roslaunch global_planning global_planning.launch"),
    ("Planning space",         "roslaunch planning_space planning_space.launch"),
    ("Behavior planning",      "roslaunch behavior_planning behavior_planning.launch"),
    ("[diffusion_planner]",    "roslaunch mid2mid_planning_diffusion mid2mid_planning_diffusion.launch  [+ /venv/diffusion_planner]"),
    (" OR [pluto]",            "roslaunch mid2mid_planning_pluto mid2mid_planning_pluto.launch           [+ /venv/pluto]"),
    ("Lateral control",        "roslaunch lateral_control lateral_control.launch"),
    ("Longitudinal control",   "roslaunch longitudinal_control longitudinal_control.launch"),
    ("Evaluation",             "roslaunch evaluation evaluation.launch mode:=carmaker"),
]

print('Launch order (one terminal per row):')
for i, (name, cmd) in enumerate(launch_sequence):
    print(f'  [{i:02d}] {name:<25}  →  {cmd}')

In [ ]:
# [SHELL] One-command launch

# Diffusion Planner — urban highway (default)
print('$ sudo -u $USER ./launch_carmaker.sh --map_loader carmaker_urban_highway')
print()
# Pluto — urban highway
print('$ sudo -u $USER ./launch_carmaker.sh --map_loader carmaker_urban_highway --model pluto')
print()
# Roundabout scenario
print('$ sudo -u $USER ./launch_carmaker.sh --map_loader carmaker_roundabout')
print('$ sudo -u $USER ./launch_carmaker.sh --map_loader carmaker_roundabout --model pluto')

---
## 6. Monitored ROS Topics

After launch, the key topics to monitor are:

In [ ]:
topics = [
    ("app/pla/planning_space",    "autohyu_msgs/PlanningSpace",   "IN  — fused perception + map context"),
    ("app/pla/trajectory",        "autohyu_msgs/Trajectory",      "OUT — ego trajectory from Diffusion Planner"),
    ("hmi/pla/trajectory_ml",     "visualization_msgs/MarkerArray","OUT — RViz trajectory visualisation"),
]

print(f"{'Topic':<40} {'Type':<35} {'Role'}")
print("-" * 100)
for topic, msg_type, role in topics:
    print(f"{topic:<40} {msg_type:<35} {role}")

---
## 7. Guidance Configuration Example (Diffusion Planner)

Collision avoidance and lane keeping can be toggled via `config/planning_ml.yaml` without any code change.  
Pluto does not use classifier guidance.

In [ ]:
import copy, yaml, os

config_path = os.path.join(os.getcwd(), "config", "planning_ml.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# --- Example: enable both guidance functions ---
cfg_with_guidance = copy.deepcopy(cfg)
cfg_with_guidance["diffusion_planner"]["use_collision_guidance"] = True
cfg_with_guidance["diffusion_planner"]["guidance_scale"] = 0.5
cfg_with_guidance["diffusion_planner"]["lane_keeping"]["enable"] = True
cfg_with_guidance["diffusion_planner"]["lane_keeping"]["scale"] = 6.0

print("Config with guidance enabled:")
print(yaml.dump(cfg_with_guidance["diffusion_planner"], default_flow_style=False))

# To apply: uncomment and re-run, then restart the Diffusion Planner node
# with open(config_path, 'w') as f:
#     yaml.dump(cfg_with_guidance, f, default_flow_style=False)

In [ ]:
# Model switching: read current model_type from config
import yaml, os

config_path = os.path.join(os.getcwd(), 'config', 'planning_ml.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

print(f"Current model_type : {cfg.get('model_type', 'diffusion_planner')}")
print()
print('To switch model, use the --model flag at launch time:')
print('  ./launch_carmaker.sh --model diffusion_planner')
print('  ./launch_carmaker.sh --model pluto')
print()
print('Pluto checkpoint path:', cfg.get('pluto', {}).get('checkpoint_path', 'N/A'))
print('Diffusion checkpoint :', cfg.get('diffusion_planner', {}).get('checkpoint_dir', 'N/A'))

---
## 8. Project Resources

| Resource | Link |
|----------|------|
| Presentation Video | *(YouTube — to be added)* |
| Demo Video | *(YouTube — to be added)* |
| Presentation Slides | [Google Slides](https://docs.google.com/presentation/d/138xGUq43BR-bDI26JRtEReK5kgTf6fG8/edit?usp=drive_link&ouid=108242632374087373637&rtpof=true&sd=true) |
| Report | [Google Drive PDF](https://drive.google.com/file/d/1JDw7noqTI9lnimkm0i9857rL1dqP2imS/view?usp=drive_link) |
| Dataset | *(Google Drive — to be added)* |

## References

```
Zheng, Y. et al. "Diffusion-Based Planning for Autonomous Driving with Flexible Guidance."
ICLR 2025. https://arxiv.org/abs/2501.15564
```